In [1]:
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.cluster import DBSCAN
from difflib import SequenceMatcher
from Levenshtein import distance as levenshtein_distance
from itertools import combinations
from collections import defaultdict
import pickle
import os
from Soccer_OCEL.role_objects import assign_roles_multi
import Soccer_OCEL.utils as utils
import pandas as pd

In [2]:
def encode_sequences(sequences: list[list[str]]) -> tuple[list[tuple], LabelEncoder]:
    le = LabelEncoder()
    all_labels = [label for seq in sequences for label in seq]
    le.fit(all_labels)
    encoded = [tuple(le.transform(seq)) for seq in sequences]
    return encoded, le


def build_distance_matrix_levenshtein(encoded_sequences: list[tuple]) -> np.ndarray:
    n = len(encoded_sequences)
    dist_matrix = np.zeros((n, n), dtype=np.float32)
    # convert to strings of tokens for levenshtein
    joined = [' '.join(map(str, seq)) for seq in encoded_sequences]
    for i, j in combinations(range(n), 2):
        d = levenshtein_distance(joined[i], joined[j])
        dist_matrix[i][j] = d
        dist_matrix[j][i] = d
    return dist_matrix


def build_distance_matrix_sequencematcher(encoded_sequences: list[tuple]) -> np.ndarray:
    n = len(encoded_sequences)
    dist_matrix = np.zeros((n, n), dtype=np.float32)
    for i, j in combinations(range(n), 2):
        sim = SequenceMatcher(None, encoded_sequences[i], encoded_sequences[j]).ratio()
        d = 1 - sim
        dist_matrix[i][j] = d
        dist_matrix[j][i] = d
    return dist_matrix


def cluster(dist_matrix: np.ndarray, eps: float, min_samples: int = 2) -> np.ndarray:
    labels = DBSCAN(eps=eps, min_samples=min_samples, metric='precomputed').fit(dist_matrix).labels_
    return labels


def inspect_clusters(labels: np.ndarray, sequences: list[list[str]]) -> dict:
    clusters = defaultdict(list)
    for idx, label in enumerate(labels):
        clusters[label].append(sequences[idx])
    return dict(clusters)


def print_clusters(clusters: dict, max_seqs: int = 3):
    for label, seqs in sorted(clusters.items()):
        tag = 'NOISE' if label == -1 else f'Cluster {label}'
        print(f"\n{tag} ({len(seqs)} sequences):")
        for s in seqs[:max_seqs]:
            print(f"  {s}")
        if len(seqs) > max_seqs:
            print(f"  ... and {len(seqs) - max_seqs} more")

In [10]:
#load all
path='data/28196177'
FD='DFL-CLU-00000P'
#games=get_games(path)
games=['J03WQQ', 'J03WR9', 'J03WPY', 'J03WOH', 'J03WOY']#Fortuna Düsseldorf
#games=['J03WQQ', 'J03WOH', 'J03WOY']#Fortuna Düsseldorf, won games
#games=['J03WR9', 'J03WPY']#Fortuna Düsseldorf lost game
GDs=[]
all_games=[]
for Game in games:
    with open(os.path.join('output','output_pkl',f"GameData_{Game}.pkl"), "rb") as f:
        GD_loaded = pickle.load(f)
    #GD_loaded.join_events(log='MOVEMENT')
    GD_loaded=assign_roles_multi(GD_loaded, team=True)
    #GD_loaded.encode_pass_direction()
    GD_loaded.encode_pass_distance()
    GD_loaded.format_log(log='ALL')
    GDs.append(GD_loaded)

In [11]:
all_games=[]
for GD_loaded in GDs:
    #GD_loaded.encode_pass_distance()
    df=GD_loaded.events.query('`case:concept:name`.notna()').copy()
    all_games.append(df)

FD_log=[]
for i, game_log in enumerate(all_games):
    teamside=GDs[i].team_sheets_df.query('tID==@FD')['Home_Away'].values[0]
    FD_log.append(game_log[game_log['case:concept:name'].str.contains(teamside, na=False)])
    #FD_log.append(game_log[(game_log['tID'] == FD) & (game_log['case:concept:name'].str.contains(teamside, na=False))])
allgame_df=pd.concat(FD_log).sort_values(["attribute:game","attribute:session", "attribute:frame"]).reset_index(drop=True)

In [5]:
all_games=[]
for GD_loaded in GDs:
    df=GD_loaded.movement_events.query('`case:concept:name`.notna()').copy()
    df['tID'] = df['Player'].apply(lambda x: utils.get_tID_from_pID(x, GD_loaded.team_sheets_df))
    all_games.append(df)

In [ ]:
all_games=[]
for GD_loaded in GDs:
    df=GD_loaded.positional_events.query('`case:concept:name`.notna()').copy()
    df['tID'] = df['Player'].apply(lambda x: utils.get_tID_from_pID(x, GD_loaded.team_sheets_df))
    all_games.append(df)

In [ ]:
FD_log=[]
for i, game_log in enumerate(all_games):
    temp=game_log.query('tID==@FD').copy()
    temp['case:concept:name']=temp['case:concept:name'] + '_' + temp['Player']
    FD_log.append(temp)
    #FD_log.append(game_log[(game_log['tID'] == FD) & (game_log['case:concept:name'].str.contains(teamside, na=False))])
allgame_df=pd.concat(FD_log).sort_values(["attribute:game","attribute:session", "attribute:frame"]).reset_index(drop=True)

In [15]:
allgame_df['n']=allgame_df['end_frame']-allgame_df['attribute:frame']
short_thr = allgame_df['n'].quantile(0.33)
long_thr  = allgame_df['n'].quantile(0.66)

def classify(d):
    if d <= short_thr:
        return "short"
    elif d <= long_thr:
        return "medium"
    else:
        return "long"

allgame_df['movement_type'] = allgame_df['n'].apply(classify)

allgame_df['concept:name'] = allgame_df['concept:name'].str.cat(allgame_df['movement_type'], sep='_')

In [16]:
grouped = allgame_df.groupby('case:concept:name')['concept:name'].apply(list)
case_ids = grouped.index.tolist()
sequences = grouped.tolist()
min_len=3
case_ids=[c for i,c in enumerate(case_ids) if len(sequences[i])>=min_len]
sequences=[s for s in sequences if len(s)>=min_len]
encoded_sequences, le = encode_sequences(sequences)

In [17]:
# levenshtein
dist_lev = build_distance_matrix_levenshtein(encoded_sequences)
labels_lev = cluster(dist_lev, eps=3, min_samples=2)
clusters_lev = inspect_clusters(labels_lev, sequences)
print("=== Levenshtein Clustering ===")
print_clusters(clusters_lev)

=== Levenshtein Clustering ===

NOISE (220 sequences):
  ['Pass_Intercepted', 'OtherBallAction', 'Play_Pass_short', 'Pass_Received', 'Play_Pass_short', 'Pass_Received', 'Play_Pass_short', 'Pass_Received', 'OtherBallAction', 'Play_Pass_medium', 'Pass_Received', 'Run', 'ShotAtGoal', 'BlockedShot']
  ['Pass_Intercepted', 'OtherBallAction', 'TacklingGame', 'Play_Pass_short', 'Pass_Received', 'TacklingGame', 'Play_Pass_short', 'Pass_Received', 'ShotAtGoal', 'SavedShot']
  ['Pass_Intercepted', 'OtherBallAction', 'Play_Pass_medium', 'Pass_Received', 'Play_Pass_medium', 'Pass_Received', 'Play_Pass_medium', 'Pass_Received', 'Play_Pass_medium', 'Pass_Received', 'Play_Pass_medium', 'Pass_Received', 'Play_Pass_medium', 'Pass_Received', 'Play_Pass_medium', 'Pass_Received', 'Play_Pass_long', 'Pass_Received', 'OtherBallAction', 'OtherBallAction', 'TacklingGame']
  ... and 217 more

Cluster 0 (356 sequences):
  ['GoalKick_Play_Pass', 'Pass_Received', 'Play_Pass_medium', 'Ball_lost']
  ['BallClaiming',

In [18]:
# sequencematcher
dist_sm = build_distance_matrix_sequencematcher(encoded_sequences)
labels_sm = cluster(dist_sm, eps=0.3, min_samples=2)
clusters_sm = inspect_clusters(labels_sm, sequences)
print("=== SequenceMatcher Clustering ===")
print_clusters(clusters_sm)

=== SequenceMatcher Clustering ===

NOISE (33 sequences):
  ['BallClaiming', 'OtherBallAction', 'RefereeBall']
  ['CornerKick_Play_Cross', 'Cross_Received', 'OtherBallAction', 'TacklingGame', 'OtherBallAction', 'TacklingGame', 'OtherBallAction', 'OtherBallAction', 'Play_Pass_long', 'Ball_lost']
  ['Pass_Intercepted', 'Offside', 'SitterPrevented']
  ... and 30 more

Cluster 0 (577 sequences):
  ['GoalKick_Play_Pass', 'Pass_Received', 'Play_Pass_medium', 'Ball_lost']
  ['ThrowIn_Play_Pass', 'Pass_Received', 'Play_Pass_medium', 'Pass_Received', 'Play_Pass_medium', 'Pass_Received', 'Play_Pass_long', 'Pass_Received', 'Play_Pass_short', 'Pass_Received', 'Play_Pass_long', 'Pass_Received', 'Play_Pass_medium', 'Pass_Received', 'Play_Cross_medium', 'Ball_lost']
  ['ThrowIn_Play_Pass', 'Pass_Received', 'Play_Pass_medium', 'Pass_Received', 'Play_Pass_medium', 'Pass_Received', 'Play_Pass_medium', 'Pass_Received', 'Play_Pass_medium', 'Ball_lost']
  ... and 574 more

Cluster 1 (5 sequences):
  ['Corn